# Produce Brain Prediction CSVs

In [1]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    sprint_dir = candidate / "codes" / "sprint"
    if (sprint_dir / "prediction_export.py").exists():
        PROJECT_ROOT = candidate
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise RuntimeError(f"Cannot find codes/sprint/prediction_export.py from {start}")

# Add legacy model path for this dataset
LEGACY_DIR = PROJECT_ROOT / "codes" / "_legacy_models" / "brain"
if str(LEGACY_DIR) not in sys.path:
    sys.path.insert(0, str(LEGACY_DIR))

from codes.sprint.prediction_export import select_least_used_cuda_before_torch_import

select_least_used_cuda_before_torch_import()

Detected CUDA devices before torch import:
  physical=0 used=630 MB / 11264 MB (5.6%) <-- selected as cuda:0


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from codes.sprint.prediction_export import export_model_specs, find_project_root

# Legacy model & dataset classes (trained with these exact architectures)
from codes._legacy_models.brain.model import Model_RNA_Only, Model_SchemeA2, Model_SchemeC2
from codes._legacy_models.brain.utils import BrainMultimodalDataset, create_diagonal_split, load_model_weights, set_seed

PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root: {PROJECT_ROOT}")

set_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Project root: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github
Using device: cuda:0


In [7]:
# ---- Paths ----
DATASET_PATH = PROJECT_ROOT / "datas" / "brain" / "Brain_Multimodal_Final_256px.h5ad"
MODEL_SAVE_ROOT = PROJECT_ROOT /"datas"/ "models" / "brain"
OUTPUT_DIR = PROJECT_ROOT /"datas"/ "outputs" / "brain"

# ---- Model specs ----
MODEL_SPECS = [
    {"label": "A2", "model_class": Model_SchemeA2,
     "model_dir_candidates": ["A220260625", "A2"],
     "output_prefix": "brain_A2"},
]
BATCH_SIZE = 32

for required_path in [DATASET_PATH, MODEL_SAVE_ROOT]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
full_dataset = BrainMultimodalDataset(str(DATASET_PATH))
_, val_dataset, _, val_loader = create_diagonal_split(
    full_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    drop_last=True,
    plot=False,
)

NUM_GENES = full_dataset.rna_data.shape[1]
NUM_TARGETS = full_dataset.protein_data.shape[1]
target_names = [str(x) for x in list(full_dataset.protein_names)]

print(f"Inference config: {NUM_GENES} genes, {NUM_TARGETS} proteins, "
      f"{len(val_dataset)} validation spots")
print(target_names)

Loading data from: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/brain/Brain_Multimodal_Final_256px.h5ad ...
Raw Images Type: <class 'numpy.ndarray'>
Raw Images Shape: (5756, 256, 256)
Using ALL Genes (No HVG filter).
RNA Input Shape: (5756, 18085)
Data Ready (Diagonal Split)!
Train: 3283 spots (Upper Triangle)
Test:  2473 spots (Lower Triangle)
Inference config: 18085 genes, 31 proteins, 2473 validation spots
['CD163-1', 'CR2-1', 'PCNA-1', 'VIM-1', 'KRT5-1', 'CD68-1', 'CEACAM8-1', 'PTPRC-1', 'HLA-DRA', 'PAX5-1', 'SDC1-1', 'PTPRC-2', 'CD8A-1', 'BCL2-1', 'CD19-1', 'PDCD1-1', 'ACTA2-1', 'FCGR3A-1', 'ITGAX-1', 'CXCR5-1', 'EPCAM-1', 'MS4A1-1', 'CD3E-1', 'CD14-1', 'CD40-1', 'PECAM1-1', 'CD4-1', 'ITGAM-1', 'CD27-1', 'CCR7-1', 'CD274-1']


In [9]:
saved_df, pred_df, target_df = export_model_specs(
    MODEL_SPECS,
    MODEL_SAVE_ROOT,
    OUTPUT_DIR,
    val_loader,
    device,
    NUM_TARGETS,
    NUM_GENES,
    target_names,
    load_model_weights,
)

display(saved_df)
display(pred_df.head())
display(target_df.head())

[A2] saved predictions: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/brain/brain_A2_predictions.csv
[A2] saved targets:     /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/brain/brain_A2_targets.csv


,Model,Weights,ModelDir,Predictions,Targets,Rows,TargetsCount
0,A2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,A2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,2473,31


,test_index,CD163-1,CR2-1,PCNA-1,VIM-1,KRT5-1,CD68-1,CEACAM8-1,PTPRC-1,HLA-DRA,...,MS4A1-1,CD3E-1,CD14-1,CD40-1,PECAM1-1,CD4-1,ITGAM-1,CD27-1,CCR7-1,CD274-1
0,0,10.499908,9.730399,9.116481,10.939980,5.831076,9.671375,7.550558,8.343777,8.713174,...,7.572442,8.600174,10.045921,9.984200,9.294926,10.302003,9.012921,9.559226,9.729317,9.335331
1,1,10.752282,9.797758,10.607371,11.410364,5.937236,10.426607,6.654271,8.982623,9.352101,...,7.265405,9.186350,9.615625,9.460798,8.566326,10.881618,8.393090,10.204681,10.343037,10.803879
2,2,10.970141,10.015121,9.821452,11.534925,6.555808,10.234204,7.908933,8.752339,9.248210,...,7.541693,9.029517,10.358271,10.052983,9.485965,10.797332,9.048106,9.906610,10.178652,9.822836
3,3,10.624410,9.763931,10.642882,11.324152,5.868765,10.510348,6.777659,9.020542,9.372939,...,7.076739,9.148885,9.655352,9.467239,8.656360,10.938928,8.441035,10.233809,10.449623,10.528441
4,4,11.300788,9.709571,10.019847,11.417897,6.282637,10.435828,6.946923,8.935440,9.062339,...,7.314238,9.032922,10.405727,9.854390,8.993799,10.818173,8.800302,9.938873,10.271685,10.035694


,test_index,CD163-1,CR2-1,PCNA-1,VIM-1,KRT5-1,CD68-1,CEACAM8-1,PTPRC-1,HLA-DRA,...,MS4A1-1,CD3E-1,CD14-1,CD40-1,PECAM1-1,CD4-1,ITGAM-1,CD27-1,CCR7-1,CD274-1
0,0,11.174988,9.853404,8.597482,11.924618,5.849325,10.106510,10.363788,9.228573,8.434464,...,8.407601,8.877661,10.164620,10.067518,9.216521,10.421835,10.533615,9.662944,9.943429,9.409191
1,1,10.948593,9.916946,10.275706,11.389141,6.154858,10.560697,7.195187,8.941937,9.579072,...,7.410952,9.275379,9.851036,9.655795,8.928773,10.901340,8.745603,10.146512,10.335335,10.716438
2,2,10.936601,9.992139,8.924390,11.964007,6.526495,10.800086,8.614864,8.687273,9.893690,...,7.383989,9.520395,10.282609,10.068409,9.188197,10.791585,8.777710,10.079707,10.304074,10.164004
3,3,10.833030,10.056423,10.741752,11.756162,6.338594,10.688986,9.111403,9.101864,9.622582,...,7.593878,9.367002,9.880782,9.934211,9.022805,10.947767,9.316141,10.295361,10.391454,10.941625
4,4,12.457991,9.864071,8.181440,12.198313,5.771441,10.284080,7.513164,8.810310,9.240191,...,7.648740,8.762802,11.110132,10.107081,9.321345,11.033033,10.167927,10.045768,10.391116,10.462217
